In [ ]:
import rebound
import numpy as np

import sys
sys.path.insert(0, '/Users/kvolk/Documents/GitHub/SBDynT/src')
import sbdynt as sbd

# Example of setting up an integration based on an orbit fit requested directly from the MPC 
While the defualt behavior of SBDynT is to request orbits from JPL's small body database and Horizons, that database lags behind the MPC

Users can instead request orbits from the MPC using the following commands. 

In [ ]:
obj_name = 'Haumea'
flag, sim, epoch = sbd.initialize_from_mpc_cartesian(
    des=obj_name, clones=10, 
    datadir="outputs-from-advanced-example-notebooks", 
    save_sbdb=True, saveic=True, planets=['outer'])

helio_orbit = sim.particles[obj_name].orbit(sim.particles[0])
print("a=", helio_orbit.a)
print("e=", helio_orbit.e)
print("i=", helio_orbit.inc)
print("node=", helio_orbit.Omega)
print("aperi=", helio_orbit.omega)
print("mean anomaly=", helio_orbit.M)

# Examples of setting up an integration based on either a list of heliocentric orbital elements or heliocentric cartesian positions and velocities 

The user can provide orbits or cartesian coordinates to initialize the small body

In [ ]:
obj_name = "example"
# Find_orb orbital elements:
epoch = 2457217.5
a = 39.4060972335126  # (semimajor axis in au)
ec = 0.19121388131996062  # (eccentricity)
inc = 0.1279988198070029  # radians! (inclination relative to the ecliptic)
node = -2.3766713138417384  # radians! (longitude of ascending node)
argperi = 3.033977870435759  # radians! (argument of pericenter)
ma = 0.15683745626851575  # radians! (mean anomaly)

flag, sim = sbd.initialize_from_heliocentric_orbit(
    des=obj_name, a=a, e=ec, inc=inc, node=node, 
    aperi=argperi, ma=ma, planets=["all"], epoch=epoch
)


helio_orbit = sim.particles[obj_name].orbit(sim.particles[0])

print("Added heliocentric orbit: ")
print("a=", helio_orbit.a)
print("e=", helio_orbit.e)
print("i=", helio_orbit.inc)
print("node=", helio_orbit.Omega)
print("aperi=", helio_orbit.omega)
print("mean anomaly=", helio_orbit.M)

For both orbital elements above and the cartesian example below, you can provide either single values or arrays of values

In [ ]:
body = 'testobject'
#pulling MPC data so I don't have to fabricate cartesian positions
flag, xv, cov_mat, epoch = sbd.get_mpc_cartesian('2015 DZ248')
#generating clones to demponstrate initializing based on a list:
x, y, z, vx, vy, vz = np.random.multivariate_normal(xv, cov_mat, 10).T
#convert from au/d to au/year
vx=vx*365.25
vy=vy*365.25
vz=vz*365.25


flag, sim = sbd.initialize_from_heliocentric_cartesian(
    des=body, 
    epoch=epoch,
    x=x, y=y, z=z, 
    vx=vx, vy=vy, vz=vz,
    planets=["outer"], 
)

helio_orbit = sim.particles['testobject_1'].orbit(sim.particles[0])

print("Added heliocentric orbit: ")
print("a=", helio_orbit.a)
print("e=", helio_orbit.e)
print("i=", helio_orbit.inc)
print("node=", helio_orbit.Omega)
print("aperi=", helio_orbit.omega)
print("mean anomaly=", helio_orbit.M)

**We also have a routine for orbits fit using Find_orb**

They use a slightly different GM for the Sun than JPL, and that's accounted for in the routine

In [ ]:
obj_name = "example"
# Find_orb orbital elements:
epoch = 2457217.5
a = 39.4060972335126  # (semimajor axis in au)
ec = 0.19121388131996062  # (eccentricity)
inc = 0.1279988198070029  # radians! (inclination relative to the ecliptic)
node = -2.3766713138417384  # radians! (longitude of ascending node)
argperi = 3.033977870435759  # radians! (argument of pericenter)
ma = 0.15683745626851575  # radians! (mean anomaly)

flag, sim = sbd.initialize_from_heliocentric_Find_Orb_orbit(
    des=obj_name, a=a, e=ec, inc=inc, node=node, aperi=argperi, ma=ma, planets=["all"], epoch=epoch
)


helio_orbit = sim.particles[obj_name].orbit(sim.particles[0])

print("Added heliocentric orbit: ")
print("a=", helio_orbit.a)
print("e=", helio_orbit.e)
print("i=", helio_orbit.inc)
print("node=", helio_orbit.Omega)
print("aperi=", helio_orbit.omega)
print("mean anomaly=", helio_orbit.M)

**Note** that there are tiny difference between the provided find_orb elements and those in the final simulation. This is because there is actually a slight difference between the masses for the Sun and planets in JPL's database and what's assumed in the find_orb code. We choose to account for this by converting the find_orb elements to heliocentric cartesian positions and velocities using find_orb's GM for the Sun and then adding the particles to the rebound simulation (which uses JPL's masses for everything) in cartesian space. Both sets of orbital elements represent the same cartesian positions and velocities. 

Additional small differences between input and output elements will occur if only a subset of the planets are included because we account for the missing planet masses. 

**NOTE**: you can initialize clones generaged by Find_Orb by passing orbital element arrays (e.g., a = [39.4,39.41,...]) rather
than single element values to ``sbd.initialize_from_heliocentric_Find_Orb_orbit``